# 3. Make a network smaller without confusing the measurements

**Prerequisite:** Lesson 2. **You will learn:** distillation, magnitude pruning, channel removal, quantization, calibration, and fair deployment comparison. Allow 25–40 minutes. We transform numbers and an untrained network first, then inspect measured study evidence.

**Execution:** Run from the repository or this folder, in order, using the installed project environment. The default lesson needs no downloads, credentials, GPU, or existing model checkpoint. Toy examples are teaching demonstrations, not the real-data study.

**Learning loop:** Read the theory, predict the output, run the example, attempt the exercise, then open the worked answer.


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/results.json').is_file())
print('Repository:', ROOT.name)


Repository: Vision


## 1. Four different operations
Distillation trains a student using a teacher's scores as well as labels. Magnitude pruning zeros small weights. Channel pruning removes entire feature channels and changes layer dimensions. Quantization represents numbers with fewer bits. These operations can be combined, but they introduce different accuracy and runtime tradeoffs.

Start with a list of eight artificial weights. Zeroing four of them changes the values but leaves eight storage positions in the dense list. A sparse storage format could exploit zeros, but that requires explicit runtime support.


In [2]:
weights = [-.9, -.02, .01, .8, -.7, .03, .04, .6]
smallest = sorted(range(len(weights)), key=lambda i: abs(weights[i]))[:4]
masked = [0.0 if i in smallest else w for i, w in enumerate(weights)]
print('Original:', weights)
print('Masked:  ', masked)
print('Dense positions before/after:', len(weights), len(masked))
assert sum(w == 0 for w in masked) == 4


Original: [-0.9, -0.02, 0.01, 0.8, -0.7, 0.03, 0.04, 0.6]
Masked:   [-0.9, 0.0, 0.0, 0.8, -0.7, 0.0, 0.0, 0.6]
Dense positions before/after: 8 8


## 2. Quantization is rounding on a limited grid
For a simplified symmetric signed INT8 example, choose a scale `s = max(abs(w)) / 127`. Store `q = round(w/s)` clamped to `[-127,127]`, then approximate with `q*s`. The approximation introduces rounding error. Real kernels also handle activations, accumulation, scales, and sometimes zero points.

PTQ collects representative activation ranges without updating weights. QAT fine-tunes with simulated quantization before conversion. Calibration must use training examples rather than the held-out test set.


In [3]:
import numpy as np
w = np.array(weights, dtype=np.float32)
scale = max(float(np.abs(w).max()) / 127, 1e-8)
q = np.clip(np.rint(w / scale), -127, 127).astype(np.int8)
restored = q.astype(np.float32) * scale
print('INT8 values:', q)
print('Maximum rounding error:', float(np.abs(w - restored).max()))
print('Raw weight-array bytes:', w.nbytes, '->', q.nbytes)
assert np.abs(w - restored).max() <= scale / 2 + 1e-6


INT8 values: [-127   -3    1  113  -99    4    6   85]
Maximum rounding error: 0.0029133856296539307
Raw weight-array bytes: 32 -> 8


## 3. Physical width changes parameter count
Construct two untrained repository networks and count their parameters. This demonstrates physical dimensions, without claiming anything about trained quality or execution speed. Halving width need not halve parameter count because some layer dimensions depend on width on both sides.


In [4]:
from edge_agent_lab.models import SensorNet
counts = {width: sum(p.numel() for p in SensorNet(width=width).parameters()) for width in [8, 4]}
print('Parameters by width:', counts)
assert counts[4] < counts[8]


Parameters by width: {8: 6422, 4: 1870}


## 4. Inspect actual deployment measurements
Now switch from toy arithmetic to the saved real-data quantization study. Every row below uses the same deployment archive format. Compare validation quality before interpreting storage and timing. p95 is the duration below which 95% of measured runs fall; it is not the slowest observed run.

Raw array bytes above exclude graphs, metadata, and packed-weight overhead. They cannot predict the exact archive reduction.


In [5]:
report = json.loads((ROOT / 'reports/quantization.json').read_text())
for candidate in report['candidates']:
    print(candidate['name'], '| archive bytes:', candidate['archive_bytes'],
          '| validation F1:', round(candidate['validation']['macro_f1'], 4),
          '| CPU batch-1 p95 ms:', round(candidate['timing']['batch1']['p95_ms'], 3))
assert report['test_used_for_selection'] is False


float_original | archive bytes: 80433 | validation F1: 0.9602 | CPU batch-1 p95 ms: 0.066
float_finetuned | archive bytes: 81299 | validation F1: 0.9602 | CPU batch-1 p95 ms: 0.066
dynamic_int8_linear | archive bytes: 58023 | validation F1: 0.9602 | CPU batch-1 p95 ms: 0.146
static_ptq_64 | archive bytes: 40627 | validation F1: 0.9575 | CPU batch-1 p95 ms: 0.06
static_ptq_512 | archive bytes: 41199 | validation F1: 0.9582 | CPU batch-1 p95 ms: 0.06
qat_int8 | archive bytes: 41199 | validation F1: 0.9575 | CPU batch-1 p95 ms: 0.06
compact_float | archive bytes: 41969 | validation F1: 0.945 | CPU batch-1 p95 ms: 0.056
compact_static_ptq_512 | archive bytes: 31343 | validation F1: 0.9462 | CPU batch-1 p95 ms: 0.05


## Exercise: choose a fair comparison
A magnitude-pruned model has 50% zeros but almost unchanged dense file size. Is pruning broken? A second model uses fewer bytes but takes longer at batch 1. Is the timing necessarily wrong?

<details><summary>Worked answer</summary>

Neither observation is contradictory. Zeroing values does not remove dense storage positions. Lower precision can add conversion and dispatch overhead that outweigh compute savings for a small workload. Compare the actual loaded deployment format, runtime, batch size, and hardware. A training checkpoint and a TorchScript deployment archive are not directly comparable storage formats.
</details>

**Application bridge:** The [post-training toolkit](../04_post_training_toolkit.ipynb) implements real pruning, calibration, conversion, and QAT with opt-in local examples. Inspect `pruning.py` and `quantization.py` next. Continue to [tool-using agents](04_tools_and_agents.ipynb).
